# NexaTel Phase 4 (part 1): Prepare the dashboard tables

Power BI works best with a few flat, ready-made tables. This notebook builds them from the cleaned Phase 1 data and saves them as CSV files in `NexaTel/dashboard_data/`. Nothing new is invented here: every column comes from the earlier phases.

| File | Rows are | Used for |
|---|---|---|
| `dim_segment.csv` | the 4 segments | the segment slicer and correct segment order |
| `customers_dash.csv` | customers | segment, tenure, state, network and complaint charts |
| `monthly_dash.csv` | month x segment | churn, retention, MRR and revenue lost trends |
| `tickets_dash.csv` | support tickets | FCR and resolution time |
| `complaints_dash.csv` | complaints | complaints by status |
| `campaigns_dash.csv` | retention offers | campaign outcomes |
| `active_subs_dash.csv` | active subscriptions | monthly charge (ARPU) distribution |
| `top_at_risk.csv` | 25 active customers | the "top at-risk customers" table |

**About the at-risk list:** it is a simple rule-based watch-list, not a prediction model. A customer scores one point for each pattern that went with higher churn in Phase 2: a Monthly plan, 2 or more unresolved complaints, a dropped-call rate in the worst quarter, and being in the Mass segment.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

BASE = "/content/drive/MyDrive/NexaTel/"
DATA = BASE + "data/"
OUT = BASE + "dashboard_data/"
os.makedirs(OUT, exist_ok=True)

customers = pd.read_csv(DATA + "customers_clean.csv", parse_dates=["acquisition_date", "churn_date"])
subs = pd.read_csv(DATA + "subscriptions_clean.csv")
subs["start_date"] = pd.to_datetime(subs["start_date"], format="%d-%m-%Y")
subs["end_date"]   = pd.to_datetime(subs["end_date"],   format="%d-%m-%Y")
tickets    = pd.read_csv(DATA + "support_tickets.csv")
complaints = pd.read_csv(DATA + "complaints.csv")
campaigns  = pd.read_csv(DATA + "retention_campaigns.csv")
network    = pd.read_csv(DATA + "network_quality.csv")
plans      = pd.read_csv(DATA + "plans.csv")
states     = pd.read_csv(DATA + "states.csv")

segment_of = customers.set_index("customer_id")["customer_segment"]
print("loaded")

Mounted at /content/drive
loaded


## 1. Segment table

In [2]:
dim_segment = pd.DataFrame({"segment": ["Mass", "Value", "Premium", "Enterprise"],
                            "segment_order": [1, 2, 3, 4]})
dim_segment.to_csv(OUT + "dim_segment.csv", index=False)
dim_segment

,segment,segment_order
0,Mass,1
1,Value,2
2,Premium,3
3,Enterprise,4


## 2. Customers table
One row per customer, with every field the charts need. Band labels start with a number (1), 2), ...) so Power BI sorts them in the right order.

In [3]:
c = customers.copy()
c["segment"] = c["customer_segment"]
c["churned"] = (c["customer_status"] == "Churned").astype(int)           # 1 = churned, 0 = active
c["billing_cycle"] = c["plan_id"].map(plans.set_index("plan_id")["billing_cycle"])
c["state_name"] = c["state_id"].map(states.set_index("state_id")["state_name"])

c["tenure_band"] = pd.cut(c["tenure_months"], bins=[0, 3, 12, 36, 1000], right=False,
                          labels=["1) 0-3 months", "2) 3-12 months", "3) 1-3 years", "4) 3+ years"])

unresolved_count = complaints[complaints["status"] == "Unresolved"].groupby("customer_id").size()
c["unresolved_complaints"] = c["customer_id"].map(unresolved_count).fillna(0).astype(int)
c["unresolved_group"] = c["unresolved_complaints"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})

avg_drop = network.groupby("customer_id")["dropped_call_rate_pct"].mean()
c["avg_dropped_call_rate"] = c["customer_id"].map(avg_drop).round(2)
c["drop_band"] = pd.qcut(c["avg_dropped_call_rate"], 4,
                         labels=["1) Lowest 25%", "2) Low-mid 25%", "3) High-mid 25%", "4) Highest 25%"])

active_charge = subs[subs["subscription_status"] == "Active"].groupby("customer_id")["monthly_charge_inr"].sum()
c["monthly_charge_active"] = c["customer_id"].map(active_charge).fillna(0)

# Rule-based watch-list score (0 to 4)
worst_quarter = c["avg_dropped_call_rate"] >= c["avg_dropped_call_rate"].quantile(0.75)
c["risk_score"] = ((c["billing_cycle"] == "Monthly").astype(int)
                   + (c["unresolved_complaints"] >= 2).astype(int)
                   + worst_quarter.astype(int)
                   + (c["segment"] == "Mass").astype(int))

customers_dash = c[["customer_id", "segment", "state_name", "city_tier", "billing_cycle", "tenure_band", "tenure_months",
                    "unresolved_group", "unresolved_complaints", "avg_dropped_call_rate", "drop_band",
                    "customer_status", "churned", "monthly_charge_active", "risk_score",
                    "acquisition_date", "churn_date"]]
customers_dash.to_csv(OUT + "customers_dash.csv", index=False)

print(customers_dash.shape, "| blanks in key columns:", customers_dash[["segment", "state_name", "billing_cycle", "tenure_band", "drop_band"]].isnull().sum().sum())
print("Churn rate by risk score (a sense-check of the watch-list rule):")
print((c.groupby("risk_score")["churned"].mean() * 100).round(1).to_string())

(19000, 17) | blanks in key columns: 0
Churn rate by risk score (a sense-check of the watch-list rule):
risk_score
0     5.4
1    17.6
2    25.7
3    33.7
4    44.9


## 3. Monthly table (month x segment)
Same method as Phase 3, repeated for each segment so the segment slicer works on the trend charts. MRR is the monthly charge of subscriptions that were live at the end of each month; revenue lost is the monthly charge of subscriptions that ended in the month.

In [4]:
months = pd.period_range("2025-04", "2026-03", freq="M")

def monthly_rows(df_customers, segment_name):
    ids = set(df_customers["customer_id"])
    seg_subs = subs[subs["customer_id"].isin(ids)]
    rows = []
    for m in months:
        month_start, month_end = m.start_time, m.end_time
        at_start = (df_customers["acquisition_date"] < month_start) & (df_customers["churn_date"].isna() | (df_customers["churn_date"] >= month_start))
        lost = at_start & (df_customers["churn_date"] >= month_start) & (df_customers["churn_date"] <= month_end)
        new = (df_customers["acquisition_date"] >= month_start) & (df_customers["acquisition_date"] <= month_end)
        end = (df_customers["acquisition_date"] <= month_end) & (df_customers["churn_date"].isna() | (df_customers["churn_date"] > month_end))

        live = seg_subs[(seg_subs["start_date"] <= month_end) & (seg_subs["end_date"].isna() | (seg_subs["end_date"] > month_end))]
        ended = seg_subs[(seg_subs["subscription_status"] == "Terminated") & (seg_subs["end_date"] >= month_start) & (seg_subs["end_date"] <= month_end)]

        rows.append({"month": m.start_time.date(), "segment": segment_name,
                     "start": int(at_start.sum()), "lost": int(lost.sum()), "new": int(new.sum()), "end": int(end.sum()),
                     "mrr_inr": float(live["monthly_charge_inr"].sum()),
                     "revenue_lost_inr": float(ended["monthly_charge_inr"].sum())})
    return rows

all_rows = []
for seg in dim_segment["segment"]:
    all_rows += monthly_rows(customers[customers["customer_segment"] == seg], seg)
monthly_dash = pd.DataFrame(all_rows)
monthly_dash.to_csv(OUT + "monthly_dash.csv", index=False)

# Check: adding the 4 segments together must give the Phase 3 numbers
check = monthly_dash.groupby("month")[["start", "lost", "new", "end", "mrr_inr", "revenue_lost_inr"]].sum()
check["churn_%"] = (check["lost"] / check["start"] * 100).round(2)
check

,start,lost,new,end,mrr_inr,revenue_lost_inr,churn_%
month,,,,,,,
2025-04-01,15515,68,99,15546,14763604.0,31972.0,0.44
2025-05-01,15546,62,113,15596,14795304.0,46627.0,0.40
2025-06-01,15596,73,108,15627,14849154.0,41164.0,0.47
2025-07-01,15627,86,125,15661,14905925.0,45647.0,0.55
2025-08-01,15661,102,125,15684,14958404.0,64720.0,0.65
2025-09-01,15684,81,113,15714,15007515.0,47574.0,0.52
2025-10-01,15714,90,135,15755,15050431.0,62214.0,0.57
2025-11-01,15755,109,113,15755,15079333.0,73763.0,0.69
2025-12-01,15755,106,106,15752,15111773.0,68683.0,0.67


## 4. Support tickets, complaints, campaigns and subscriptions

In [5]:
tickets_dash = tickets.copy()
tickets_dash["segment"] = tickets_dash["customer_id"].map(segment_of)
tickets_dash["fcr"] = (tickets_dash["first_contact_resolution"] == "Yes").astype(int)
tickets_dash = tickets_dash[["ticket_id", "customer_id", "segment", "category", "priority", "channel", "status", "fcr", "resolution_hours"]]
tickets_dash.to_csv(OUT + "tickets_dash.csv", index=False)

complaints_dash = complaints.copy()
complaints_dash["segment"] = complaints_dash["customer_id"].map(segment_of)
complaints_dash = complaints_dash[["complaint_id", "customer_id", "segment", "complaint_type", "severity", "status"]]
complaints_dash.to_csv(OUT + "complaints_dash.csv", index=False)

campaigns_dash = campaigns.copy()
campaigns_dash["segment"] = campaigns_dash["customer_id"].map(segment_of)
campaigns_dash["retained"] = (campaigns_dash["retained_flag"] == "Yes").astype(int)
campaigns_dash = campaigns_dash[["campaign_id", "customer_id", "segment", "campaign_name", "offer_type", "channel", "response", "retained"]]
campaigns_dash.to_csv(OUT + "campaigns_dash.csv", index=False)

active_subs_dash = subs[subs["subscription_status"] == "Active"].copy()
active_subs_dash["segment"] = active_subs_dash["customer_id"].map(segment_of)
active_subs_dash["charge_band"] = pd.cut(active_subs_dash["monthly_charge_inr"], right=False,
    bins=[0, 250, 500, 750, 1000, 1500, 2000, 3000, 100000],
    labels=["1) under 250", "2) 250-499", "3) 500-749", "4) 750-999", "5) 1000-1499", "6) 1500-1999", "7) 2000-2999", "8) 3000+"])
active_subs_dash = active_subs_dash[["subscription_id", "customer_id", "segment", "billing_cycle", "monthly_charge_inr", "charge_band"]]
active_subs_dash.to_csv(OUT + "active_subs_dash.csv", index=False)

print("tickets:", len(tickets_dash), "| complaints:", len(complaints_dash), "| campaigns:", len(campaigns_dash), "| active subscriptions:", len(active_subs_dash))

tickets: 31490 | complaints: 16584 | campaigns: 6475 | active subscriptions: 17017


## 5. Top 25 at-risk customers (active customers only)

In [6]:
top_at_risk = (c[c["churned"] == 0]
               .sort_values(["risk_score", "monthly_charge_active"], ascending=False)
               .head(25)[["customer_id", "segment", "state_name", "billing_cycle", "tenure_months",
                          "unresolved_complaints", "avg_dropped_call_rate", "risk_score", "monthly_charge_active"]])
top_at_risk.to_csv(OUT + "top_at_risk.csv", index=False)

print("Active customers by risk score:")
print(c[c["churned"] == 0]["risk_score"].value_counts().sort_index().to_string())
top_at_risk

Active customers by risk score:
risk_score
0    3392
1    7558
2    3837
3     669
4      38


,customer_id,segment,state_name,billing_cycle,tenure_months,unresolved_complaints,avg_dropped_call_rate,risk_score,monthly_charge_active
8017,C0008018,Mass,West Bengal,Monthly,28,2,4.98,4,1199.0
3378,C0003379,Mass,Jammu and Kashmir,Monthly,63,2,3.27,4,998.0
2555,C0002556,Mass,West Bengal,Monthly,104,2,3.26,4,299.0
3782,C0003783,Mass,West Bengal,Monthly,104,3,3.47,4,299.0
4303,C0004304,Mass,Andhra Pradesh,Monthly,48,2,3.66,4,299.0
4761,C0004762,Mass,Jammu and Kashmir,Monthly,115,2,4.36,4,299.0
5007,C0005008,Mass,Rajasthan,Monthly,77,2,4.30,4,299.0
5392,C0005393,Mass,Gujarat,Monthly,58,2,4.61,4,299.0
6868,C0006869,Mass,Uttar Pradesh,Monthly,69,2,6.24,4,299.0
8674,C0008675,Mass,Punjab,Monthly,126,3,4.44,4,299.0


## 6. Final check: all files saved

In [7]:
for f in sorted(os.listdir(OUT)):
    print(f, "-", len(pd.read_csv(OUT + f)), "rows")

active_subs_dash.csv - 17017 rows
campaigns_dash.csv - 6475 rows
complaints_dash.csv - 16584 rows
customers_dash.csv - 19000 rows
dim_segment.csv - 4 rows
monthly_dash.csv - 48 rows
tickets_dash.csv - 31490 rows
top_at_risk.csv - 25 rows
